Análise de Negócios

Com base nos preços dos combustíveis ofertados nos postos

Fomos contratados por uma Petroleira para construir uma análise de negócios para entender a dinâmica de preços que está sendo ofertado pelos postos.
1 - Teremos que trabalhar com Dados Existentes
link: https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/serie-historica-de-precos-de-combustiveis


Como foi uma solicitação do nosso cliente, teremos que avaliar alguns anos da base de dados.

Quando abrimos as tabelas nos deparamos que cada semestre possui uma quantidade muito grande de dados, logo juntas todas as tabelas fica difícil!

Como muitas das vezes não temos o Engenheiro de Dados, temos que dar os nossos pulos!

2 - Para Armazenar esses dados em um banco de dados escolhemos o PostgreSQL versão 17 baixada do site 
https://www.postgresql.org/download/

Instalando as bibliotecas

In [3]:
import pandas as pd
import pandas.io.sql as sqlio
import psycopg2 as ps

import warnings
warnings.filterwarnings('ignore')

In [7]:
import os

# As credenciais vêm de variáveis de ambiente (veja o arquivo .env.example).
# Nunca escreva a senha diretamente no notebook.
conn = ps.connect(dbname=os.getenv('DB_NAME', 'ANP'),
                  user=os.getenv('DB_USER', 'postgres'),
                  password=os.environ['DB_PASSWORD'],
                  host=os.getenv('DB_HOST', 'localhost'),
                  port=os.getenv('DB_PORT', '5432'))

In [8]:
sql = """
SELECT * FROM anp.preco_combustivel
"""

In [9]:
df = sqlio.read_sql_query(sql, conn)

In [10]:
df.head()

,regiao,estado,municipio,revenda,cnpj,nome_rua,numero_rua,complemento,bairro,cep,produto,data_coleta,valor_venda,unidade_medida,bandeira
0,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,NaN,FEITORIA,93048-000,GASOLINA,2021-01-01,4.599,R$ / litro,BRANCA
1,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,NaN,FEITORIA,93048-000,ETANOL,2021-01-01,4.199,R$ / litro,BRANCA
2,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,NaN,FEITORIA,93048-000,GASOLINA ADITIVADA,2021-01-01,4.799,R$ / litro,BRANCA
3,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,NaN,FEITORIA,93048-000,DIESEL,2021-01-01,3.499,R$ / litro,BRANCA
4,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,NaN,FEITORIA,93048-000,DIESEL S10,2021-01-01,3.599,R$ / litro,BRANCA


In [11]:
print(f'Quantidade de linhas e colunas {df.shape}')

Quantidade de linhas e colunas (4322192, 15)


Tipo das Colunas

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4322192 entries, 0 to 4322191
Data columns (total 15 columns):
 #   Column          Dtype  
---  ------          -----  
 0   regiao          str    
 1   estado          str    
 2   municipio       str    
 3   revenda         str    
 4   cnpj            str    
 5   nome_rua        str    
 6   numero_rua      str    
 7   complemento     str    
 8   bairro          str    
 9   cep             str    
 10  produto         str    
 11  data_coleta     object 
 12  valor_venda     float64
 13  unidade_medida  str    
 14  bandeira        str    
dtypes: float64(1), object(1), str(13)
memory usage: 494.6+ MB


In [13]:
df['data_coleta']= pd.to_datetime(df['data_coleta'])

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4322192 entries, 0 to 4322191
Data columns (total 15 columns):
 #   Column          Dtype        
---  ------          -----        
 0   regiao          str          
 1   estado          str          
 2   municipio       str          
 3   revenda         str          
 4   cnpj            str          
 5   nome_rua        str          
 6   numero_rua      str          
 7   complemento     str          
 8   bairro          str          
 9   cep             str          
 10  produto         str          
 11  data_coleta     datetime64[s]
 12  valor_venda     float64      
 13  unidade_medida  str          
 14  bandeira        str          
dtypes: datetime64[s](1), float64(1), str(13)
memory usage: 494.6 MB


Verificar se há nulos nas colunas

In [15]:
df.isnull().sum()

regiao                  0
estado                  0
municipio               0
revenda                 0
cnpj                    0
nome_rua                0
numero_rua           1202
complemento       3370250
bairro               8710
cep                     0
produto                 0
data_coleta             0
valor_venda             0
unidade_medida          0
bandeira                0
dtype: int64

Selecionar as colunas que vamos trabalhar

In [16]:
df_anp = df[['data_coleta', 'regiao', 'estado', 'municipio', 'bandeira','produto', 'valor_venda']]

In [17]:
df_anp.head()

,data_coleta,regiao,estado,municipio,bandeira,produto,valor_venda
0,2021-01-01,S,RS,SAO LEOPOLDO,BRANCA,GASOLINA,4.599
1,2021-01-01,S,RS,SAO LEOPOLDO,BRANCA,ETANOL,4.199
2,2021-01-01,S,RS,SAO LEOPOLDO,BRANCA,GASOLINA ADITIVADA,4.799
3,2021-01-01,S,RS,SAO LEOPOLDO,BRANCA,DIESEL,3.499
4,2021-01-01,S,RS,SAO LEOPOLDO,BRANCA,DIESEL S10,3.599


In [18]:
df_anp['ano'] = df_anp['data_coleta'].dt.year
df_anp['mes'] = df_anp['data_coleta'].dt.month

In [19]:
df_anp.head()

,data_coleta,regiao,estado,municipio,bandeira,produto,valor_venda,ano,mes
0,2021-01-01,S,RS,SAO LEOPOLDO,BRANCA,GASOLINA,4.599,2021,1
1,2021-01-01,S,RS,SAO LEOPOLDO,BRANCA,ETANOL,4.199,2021,1
2,2021-01-01,S,RS,SAO LEOPOLDO,BRANCA,GASOLINA ADITIVADA,4.799,2021,1
3,2021-01-01,S,RS,SAO LEOPOLDO,BRANCA,DIESEL,3.499,2021,1
4,2021-01-01,S,RS,SAO LEOPOLDO,BRANCA,DIESEL S10,3.599,2021,1


Estatística Básica

In [20]:
df_anp.describe().round(2)

,data_coleta,valor_venda,ano,mes
count,4322192,4322192.00,4322192.00,4322192.00
mean,2023-06-29 09:50:56,5.62,2023.00,6.47
min,2021-01-01 00:00:00,2.05,2021.00,1.00
25%,2022-04-11 00:00:00,4.91,2022.00,4.00
50%,2023-07-05 00:00:00,5.79,2023.00,6.00
75%,2024-09-02 00:00:00,6.29,2024.00,9.00
max,2025-12-31 00:00:00,9.79,2025.00,12.00
std,NaN,1.01,1.38,3.39


Quais são os tipos de produtos que são comercializados?

In [21]:
#Visualizando os valores únicos
print(f'Os produtos conecializados{df_anp.produto.unique()}')

Os produtos conecializados<StringArray>
['GASOLINA', 'ETANOL', 'GASOLINA ADITIVADA', 'DIESEL', 'DIESEL S10', 'GNV']
Length: 6, dtype: str


Quais anos e meses que estão na Base?

In [22]:
#Visualizando os valores únicos
print(f'Os anos da Base: {df_anp.ano.unique()}')
print(f'Os meses da Base: {df_anp.mes.unique()}')

Os anos da Base: [2021 2022 2023 2024 2025]
Os meses da Base: [ 1  4  2  3  7  6  5  8  9 10 11 12]


Descobrindo os valores mínimos, máximos e médios dos produtos por ano

In [23]:
df_anp_valor = df_anp[['ano', 'produto', 'valor_venda']]

In [24]:
df_anp_valor.groupby(['produto','ano']).agg(['min', 'max', 'mean']).round(2)

valor_venda            
                                min   max  mean
produto            ano                         
DIESEL             2021        3.10  6.99  4.69
                   2022        3.14  9.00  6.63
                   2023        3.97  7.99  5.76
                   2024        4.69  8.19  5.97
                   2025        5.24  8.39  6.15
DIESEL S10         2021        2.80  6.96  4.74
                   2022        3.59  9.65  6.73
                   2023        4.19  9.00  5.86
                   2024        4.99  8.99  6.03
                   2025        4.73  9.29  6.21
ETANOL             2021        2.05  7.90  4.67
                   2022        2.49  7.98  4.85
                   2023        2.69  6.96  4.00
                   2024        2.63  7.35  4.07
                   2025        3.09  7.44  4.46
GASOLINA           2021        3.10  8.00  5.89
                   2022        3.49  8.99  6.32
                   2023        4.09  8.19  5.53
                   2024        4.55  7.99  5.93
                   2025        4.99  9.29  6.25
GASOLINA ADITIVADA 2021        3.46  8.99  6.04
                   2022        4.09  9.28  6.47
                   2023        3.47  9.79  5.71
                   2024        4.69  9.79  6.12
                   2025        5.19  9.59  6.45
GNV                2021        2.38  6.70  3.89
                   2022        3.17  7.99  5.04
                   2023        2.39  6.71  4.64
                   2024        3.54  6.89  4.77
                   2025        3.39  6.49  4.74

In [25]:
df_anp_valor_estado = df_anp[['ano', 'produto', 'estado','valor_venda']]

In [26]:
df_anp_valor_estado.groupby(['produto','ano','estado']).agg(['min', 'max', 'mean']).round(2)

valor_venda            
                            min   max  mean
produto ano  estado                        
DIESEL  2021 AC            4.30  6.85  5.61
             AL            3.65  6.30  4.93
             AM            3.38  6.30  4.66
             AP            4.00  5.77  4.75
             BA            3.45  6.39  4.75
...                         ...   ...   ...
GNV     2025 RN            4.59  6.19  5.01
             RS            4.29  6.14  4.82
             SC            4.39  5.54  5.07
             SE            4.69  4.99  4.85
             SP            3.39  6.29  4.66

[771 rows x 3 columns]

Descobrir a quantidade de bandeiras por produto/ano

In [27]:
df_anp_bandeira = df_anp[['ano','bandeira','produto']]

In [28]:
df_anp_bandeira.groupby(['ano','produto']).bandeira.nunique()

ano   produto           
2021  DIESEL                54
      DIESEL S10            54
      ETANOL                55
      GASOLINA              55
      GASOLINA ADITIVADA    54
      GNV                   22
2022  DIESEL                51
      DIESEL S10            60
      ETANOL                60
      GASOLINA              60
      GASOLINA ADITIVADA    58
      GNV                   24
2023  DIESEL                43
      DIESEL S10            46
      ETANOL                47
      GASOLINA              48
      GASOLINA ADITIVADA    47
      GNV                   22
2024  DIESEL                47
      DIESEL S10            46
      ETANOL                49
      GASOLINA              51
      GASOLINA ADITIVADA    47
      GNV                   22
2025  DIESEL                45
      DIESEL S10            47
      ETANOL                48
      GASOLINA              49
      GASOLINA ADITIVADA    47
      GNV                   18
Name: bandeira, dtype: int64

In [29]:
df_anp_bandeira_estado = df_anp[['ano','bandeira','produto', 'estado']]

In [30]:
df_anp_bandeira_estado.groupby(['ano','produto','estado']).bandeira.nunique()

ano   produto  estado
2021  DIESEL   AC        7
               AL        6
               AM        9
               AP        4
               BA        9
                        ..
2025  GNV      RN        5
               RS        5
               SC        7
               SE        3
               SP        5
Name: bandeira, Length: 771, dtype: int64

Vamos usar o formato Parquet para exportar o dataframe criado

In [33]:
import os

In [38]:
# Salva a base tratada em Parquet (compacto e rápido de carregar)
# (caminho relativo à pasta notebooks/, onde este notebook está)
os.makedirs('../data/processed', exist_ok=True)
df_anp.to_parquet('../data/processed/dados_anp_tratados.parquet', index=False)
print("Base salva com sucesso!")

Base salva com sucesso!
